# Code for fast adaptation and save the weights in right format

In [1]:
from pathlib import Path
import numpy as np
from pynwb import NWBHDF5IO
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import os
import pickle
import torch
import time as tm
from torch.autograd import grad as autograd_grad
from torch.nn.utils.rnn import pad_sequence
from torch.func import functional_call

from neural_decoder.maml_align import MAMLAligner

In [2]:
device = "cuda"
original_model_base = "out/"
save_path = "../falcon_submit/maml_3gram/linear_long_25/"

os.makedirs(save_path, exist_ok=True)

In [3]:
original_model = "run_20260417-124107" # 2layer, 100,000 steps

calib = False # True if local test, False if online, drop the first 2 sentences or not

is_last = True # Use last checkpoint or lowest validation CER model
inner_step = 25 # for fast adaptation, we use 25 inner gradient steps

k_shot = 3 # number of samples for fast adaptation

## Preperation

### Load the calibration data

In [4]:
data_dir = Path("../data/000950/")
if not data_dir.exists():
    raise FileNotFoundError("Data not found. Please make sure to run data setup first (e.g. ./download_falcon_datasets.sh).")

calib_path_local = Path(data_dir / 'sub-T5-held-in-calib')
calib_path_remote = Path(data_dir / 'sub-T5-held-out-calib')

calib_files_local = sorted(calib_path_local.glob('*.nwb'))
calib_files_remote = sorted(calib_path_remote.glob('*.nwb'))

In [5]:
calib_files = calib_files_local + calib_files_remote
calib_files

[PosixPath('../data/000950/sub-T5-held-in-calib/sub-T5-held-in-calib_ses-20220518.nwb'),
 PosixPath('../data/000950/sub-T5-held-in-calib/sub-T5-held-in-calib_ses-20220523.nwb'),
 PosixPath('../data/000950/sub-T5-held-in-calib/sub-T5-held-in-calib_ses-20220525.nwb'),
 PosixPath('../data/000950/sub-T5-held-in-calib/sub-T5-held-in-calib_ses-20220601.nwb'),
 PosixPath('../data/000950/sub-T5-held-in-calib/sub-T5-held-in-calib_ses-20220603.nwb'),
 PosixPath('../data/000950/sub-T5-held-in-calib/sub-T5-held-in-calib_ses-20220606.nwb'),
 PosixPath('../data/000950/sub-T5-held-in-calib/sub-T5-held-in-calib_ses-20220608.nwb'),
 PosixPath('../data/000950/sub-T5-held-in-calib/sub-T5-held-in-calib_ses-20220613.nwb'),
 PosixPath('../data/000950/sub-T5-held-in-calib/sub-T5-held-in-calib_ses-20220615.nwb'),
 PosixPath('../data/000950/sub-T5-held-in-calib/sub-T5-held-in-calib_ses-20220622.nwb'),
 PosixPath('../data/000950/sub-T5-held-in-calib/sub-T5-held-in-calib_ses-20220901.nwb'),
 PosixPath('../data/0

In [6]:
def load_nwb(fn: str):
    print(f'Loading {fn}')
    with NWBHDF5IO(fn, 'r') as io:
        nwbfile = io.read()

        trial_info = (
            nwbfile.trials.to_dataframe()
            .reset_index()
        )

        binned_spikes = nwbfile.acquisition['binned_spikes'].data[()]
        time = nwbfile.acquisition['binned_spikes'].timestamps[()]
        eval_mask = nwbfile.acquisition['eval_mask'].data[()].astype(bool)

    return(
        binned_spikes,
        time,
        trial_info,
        eval_mask
    )

### Record the mean and average

In [7]:
# hashed by sessions
all_avg = {}
all_std = {}

In [8]:
def hash_dataset(handle):
        r"""
            handle - path.stem of a datafile.
            Convenience function to help identify what "session" a datafile belongs to.. If multiple files per session in real-world time, this may _not_ uniquely identify runfile.
        """
        if isinstance(handle, Path):
            handle = handle.stem
        else:
              raise TypeError("input should be a datafile!")
        return handle.split('_')[1]

In [9]:
def h2_relabel_hotfix(dandi_tag):
    if dandi_tag.startswith('ses'):
        evaluator_tag = dandi_tag.split('-')[1]
        evaluator_tag = evaluator_tag[:4] + '.' + evaluator_tag[4:6] + '.' + evaluator_tag[6:]
        return evaluator_tag
    return dandi_tag

In [10]:
for fn in calib_files:
    spikes, time, tinfo, eval_mask = load_nwb(fn)
    dataset_tag = hash_dataset(fn)
    dataset_tag = h2_relabel_hotfix(dataset_tag)
    feats_mean = np.mean(spikes, axis=0, keepdims=True)
    feats_std = np.std(spikes, axis=0, keepdims=True)
    all_avg[dataset_tag] = feats_mean
    all_std[dataset_tag] = feats_std

Loading ../data/000950/sub-T5-held-in-calib/sub-T5-held-in-calib_ses-20220518.nwb
Loading ../data/000950/sub-T5-held-in-calib/sub-T5-held-in-calib_ses-20220523.nwb
Loading ../data/000950/sub-T5-held-in-calib/sub-T5-held-in-calib_ses-20220525.nwb
Loading ../data/000950/sub-T5-held-in-calib/sub-T5-held-in-calib_ses-20220601.nwb
Loading ../data/000950/sub-T5-held-in-calib/sub-T5-held-in-calib_ses-20220603.nwb
Loading ../data/000950/sub-T5-held-in-calib/sub-T5-held-in-calib_ses-20220606.nwb
Loading ../data/000950/sub-T5-held-in-calib/sub-T5-held-in-calib_ses-20220608.nwb
Loading ../data/000950/sub-T5-held-in-calib/sub-T5-held-in-calib_ses-20220613.nwb
Loading ../data/000950/sub-T5-held-in-calib/sub-T5-held-in-calib_ses-20220615.nwb
Loading ../data/000950/sub-T5-held-in-calib/sub-T5-held-in-calib_ses-20220622.nwb
Loading ../data/000950/sub-T5-held-in-calib/sub-T5-held-in-calib_ses-20220901.nwb
Loading ../data/000950/sub-T5-held-in-calib/sub-T5-held-in-calib_ses-20220929.nwb
Loading ../data/

In [11]:
avg_and_std = {}
avg_and_std['avg'] = all_avg
avg_and_std['std'] = all_std
with open(save_path+'avg_std', 'wb') as handle:
    pickle.dump(avg_and_std, handle)

### Read features from the data

Spikes are recorded into one single array, we need to find the spike for each trial.

In [12]:
def loadFeatures(sessionPath):
    spikes, time, tinfo, eval_mask = load_nwb(sessionPath)
    dataset_tag = hash_dataset(fn)
    dataset_tag = h2_relabel_hotfix(dataset_tag)

    input_features = []
    transcriptions = []
    frame_lens = []

    local_avg = all_avg[dataset_tag]
    local_std = all_std[dataset_tag]

    for _, row in tinfo.iterrows():
    # find the spikes for each trial
        trail_start = row['start_time']
        trail_end = row['stop_time']
        start_id = np.where(time==trail_start)[0][0]
        end_id = np.where(time==trail_end)[0][0]
        features = spikes[start_id:end_id+1]
        sentence_len = features.shape[0]
        sentence = row['cue']

        # normalization
        features = (features - local_avg) / (local_std + 1e-8)

        input_features.append(features)
        transcriptions.append(sentence)
        frame_lens.append(sentence_len)

    #convert to tfRecord file
    session_data = {
        'inputFeatures': input_features,
        'transcriptions': transcriptions,
        'frameLens': frame_lens
    }

    return session_data

## Data preprocessing

- Use char id

In [13]:
CHAR_DEF = ['a', 'b', 'c', 'd', 'e', 'f', 'g',
            'h', 'i', 'j', 'k', 'l', 'm', 'n',
            'o', 'p', 'q', 'r', 's', 't',
            'u', 'v', 'w', 'x', 'y', 'z',
            '>', ',', "'", '~', '?']

def charToId(c):
    return CHAR_DEF.index(c)

In [14]:
def getDataset(fileName, calibration=True):
    session_data = loadFeatures(fileName)
        
    allDat = []
    trueSentences = []
    seqElements = []

    if calibration: # first 2 sentences are the same as minival dataset
        trail_range = range(2, len(session_data['inputFeatures']))
    else:
        trail_range = range(len(session_data['inputFeatures']))

    # for each trial
    for x in trail_range:
        allDat.append(session_data['inputFeatures'][x])
        trueSen = session_data['transcriptions'][x]
        trueSentences.append(trueSen)

        maxSeqLen = 200 # max char len is 117
        seqClassIDs = np.zeros([maxSeqLen]).astype(np.int32)
        seqLen = len(trueSen)
        seqClassIDs[0:seqLen] = [charToId(c) + 1 for c in trueSen]
        seqElements.append(seqClassIDs)

    newDataset = {}
    newDataset['sentenceDat'] = allDat
    newDataset['transcriptions'] = trueSentences
    newDataset['chars'] = seqElements

    charLens = []
    for x in range(len(newDataset['sentenceDat'])):
        zeroIdx = np.argwhere(newDataset['chars'][x]==0)
        charLens.append(zeroIdx[0,0])
    
    newDataset['charLens'] = np.array(charLens)
    
    return newDataset

In [15]:
calibDatasets = {}
for fn in calib_files:
    calibDataset = getDataset(fn, calibration=calib)
    dataset_tag = hash_dataset(fn)
    dataset_tag = h2_relabel_hotfix(dataset_tag)
    calibDatasets[dataset_tag] = calibDataset

Loading ../data/000950/sub-T5-held-in-calib/sub-T5-held-in-calib_ses-20220518.nwb
Loading ../data/000950/sub-T5-held-in-calib/sub-T5-held-in-calib_ses-20220523.nwb
Loading ../data/000950/sub-T5-held-in-calib/sub-T5-held-in-calib_ses-20220525.nwb
Loading ../data/000950/sub-T5-held-in-calib/sub-T5-held-in-calib_ses-20220601.nwb
Loading ../data/000950/sub-T5-held-in-calib/sub-T5-held-in-calib_ses-20220603.nwb
Loading ../data/000950/sub-T5-held-in-calib/sub-T5-held-in-calib_ses-20220606.nwb
Loading ../data/000950/sub-T5-held-in-calib/sub-T5-held-in-calib_ses-20220608.nwb
Loading ../data/000950/sub-T5-held-in-calib/sub-T5-held-in-calib_ses-20220613.nwb
Loading ../data/000950/sub-T5-held-in-calib/sub-T5-held-in-calib_ses-20220615.nwb
Loading ../data/000950/sub-T5-held-in-calib/sub-T5-held-in-calib_ses-20220622.nwb
Loading ../data/000950/sub-T5-held-in-calib/sub-T5-held-in-calib_ses-20220901.nwb
Loading ../data/000950/sub-T5-held-in-calib/sub-T5-held-in-calib_ses-20220929.nwb
Loading ../data/

In [16]:
calibDatasets.keys()

dict_keys(['2022.05.18', '2022.05.23', '2022.05.25', '2022.06.01', '2022.06.03', '2022.06.06', '2022.06.08', '2022.06.13', '2022.06.15', '2022.06.22', '2022.09.01', '2022.09.29', '2022.10.06', '2022.10.18', '2022.10.25', '2022.10.27', '2022.11.01', '2022.11.03', '2022.12.08', '2022.12.15', '2023.02.28', '2023.04.17', '2023.05.31', '2023.06.28', '2023.08.16', '2023.10.09'])

In [17]:
heldout_dates = ['2023.04.17', '2023.05.31', '2023.06.28', '2023.08.16', '2023.10.09']

In [18]:
def _pad_samples(features):
    """Pad variable-length sequences within a task"""
    return pad_sequence(features, batch_first=True, padding_value=0)

In [19]:
support_sets = {}
for key, day_data in calibDatasets.items():
    # Convert to tensors first
    if key in heldout_dates:
        print(f"Using {k_shot}-shot for {key}")
        neural_feats = [torch.as_tensor(x) for x in day_data["sentenceDat"]][:k_shot]
        chars = [torch.as_tensor(x) for x in day_data["chars"]][:k_shot]
        char_lens = torch.tensor(day_data["charLens"][:k_shot])
    else:
        neural_feats = [torch.as_tensor(x) for x in day_data["sentenceDat"]]
        chars = [torch.as_tensor(x) for x in day_data["chars"]]
        char_lens = torch.tensor(day_data["charLens"])

    print (f"Dataset {key}:")
    print (f"  Number of samples: {len(neural_feats)}")
    print (f"  Neural feature shapes: {[x.shape for x in neural_feats]}")
    print (f"  Char sequence shapes: {[x.shape for x in chars]}")

    support = (
        _pad_samples(neural_feats),
        _pad_samples(chars),
        torch.tensor([x.shape[0] for x in neural_feats]),
        char_lens,
    )
    support_sets[key] = support

Dataset 2022.05.18:
  Number of samples: 30
  Neural feature shapes: [torch.Size([2687, 192]), torch.Size([2536, 192]), torch.Size([982, 192]), torch.Size([1927, 192]), torch.Size([2205, 192]), torch.Size([2473, 192]), torch.Size([3158, 192]), torch.Size([3013, 192]), torch.Size([1305, 192]), torch.Size([2608, 192]), torch.Size([742, 192]), torch.Size([715, 192]), torch.Size([2591, 192]), torch.Size([2586, 192]), torch.Size([1191, 192]), torch.Size([816, 192]), torch.Size([3206, 192]), torch.Size([2766, 192]), torch.Size([2690, 192]), torch.Size([915, 192]), torch.Size([3209, 192]), torch.Size([977, 192]), torch.Size([3365, 192]), torch.Size([3422, 192]), torch.Size([1021, 192]), torch.Size([3673, 192]), torch.Size([1941, 192]), torch.Size([2709, 192]), torch.Size([969, 192]), torch.Size([943, 192])]
  Char sequence shapes: [torch.Size([200]), torch.Size([200]), torch.Size([200]), torch.Size([200]), torch.Size([200]), torch.Size([200]), torch.Size([200]), torch.Size([200]), torch.Size(

In [20]:
def _to_device(data, device):
    if isinstance(data, dict):
        return {k: _to_device(v, device) for k, v in data.items()}
    elif isinstance(data, (list, tuple)):
        return type(data)(_to_device(v, device) for v in data)
    elif torch.is_tensor(data):
        return data.to(device)
    else:
        return data

In [21]:
support_batch = _to_device(support_sets, device)

## Load model

In [22]:
# def initModel(modelPath, device="cuda"):
#     with open(os.path.join(modelPath, "args"), "rb") as handle:
#         args = pickle.load(handle)

#     if "single_layer" not in args.keys():
#         print("Single layer")
#         single_layer = True
#         align_hidden = None
#     else:
#         single_layer = args["single_layer"]
#         align_hidden = args["align_hidden"]

#     if "inner_lr" not in args.keys():
#         print("Using default inner_lr = 0.07, same as MAML aligner")
#         inner_lr = 0.07
#     else:
#         inner_lr = args["inner_lr"]

#     if "inner_steps" not in args.keys():
#         inner_steps = 3
#     else:        
#         inner_steps = args["inner_steps"]
    
#     model = MAMLAligner(
#         pretrain_model=args["pretrain"],
#         neural_dim=args["nInputFeatures"],
#         n_classes=args["nClasses"],
#         hidden_dim=args["nUnits"],
#         layer_dim=args["nLayers"],
#         dropout=0, # close dropout when fast adaptation
#         strideLen=args["strideLen"],
#         kernelLen=args["kernelLen"],
#         gaussianSmoothWidth=args["gaussianSmoothWidth"],
#         whiteNoiseSD=args["whiteNoiseSD"], 
#         constantOffsetSD=args["constantOffsetSD"],
#         bidirectional=args["bidirectional"],
#         first_order=True,
#         inner_lr=inner_lr,
#         outer_lr=args["outer_lr"],
#         optimizer_type=args["optimizer_type"],
#         use_scheduler=args["use_scheduler"],
#         l2_decay=args["l2_decay"], 
#         momentum=args["momentum"],
#         nesterov=args["nesterov"],
#         gamma=args["gamma"],
#         stepSize=args["stepSize"],
#         inner_steps=inner_steps,
#         single_layer=single_layer,
#         align_hidden=align_hidden,
#     ).to(device)
    
#     return model

In [23]:
def loadModel(modelPath, device="cuda", last=False):
    # load pl model
    if last:
        pl_model = torch.load(modelPath+"/last.ckpt", map_location=device)
    else:
        pl_model = torch.load(modelPath+"/modelWeights", map_location=device)
    
    # load hyperparameters
    with open(os.path.join(modelPath, "args"), "rb") as handle:
        args = pickle.load(handle)
    state_dict = pl_model["state_dict"]

    if "single_layer" not in args.keys():
        print("Single layer")
        single_layer = True
        align_hidden = None
    else:
        single_layer = args["single_layer"]
        align_hidden = args["align_hidden"]

    if "inner_lr" not in args.keys():
        print("Using default inner_lr = 0.07, same as MAML aligner")
        inner_lr = 0.07
    else:
        inner_lr = args["inner_lr"]

    if "inner_steps" not in args.keys():
        inner_steps = 3
    else:        
        inner_steps = args["inner_steps"]
    
    model = MAMLAligner(
        pretrain_model=args["pretrain"],
        neural_dim=args["nInputFeatures"],
        n_classes=args["nClasses"],
        hidden_dim=args["nUnits"],
        layer_dim=args["nLayers"],
        dropout=0, # close dropout when fast adaptation
        strideLen=args["strideLen"],
        kernelLen=args["kernelLen"],
        gaussianSmoothWidth=args["gaussianSmoothWidth"],
        whiteNoiseSD=args["whiteNoiseSD"], 
        constantOffsetSD=args["constantOffsetSD"],
        bidirectional=args["bidirectional"],
        first_order=True,
        inner_lr=inner_lr,
        outer_lr=args["outer_lr"],
        optimizer_type=args["optimizer_type"],
        use_scheduler=args["use_scheduler"],
        l2_decay=args["l2_decay"], 
        momentum=args["momentum"],
        nesterov=args["nesterov"],
        gamma=args["gamma"],
        stepSize=args["stepSize"],
        inner_steps=inner_steps,
        single_layer=single_layer,
        align_hidden=align_hidden,
    ).to(device)

    model.load_state_dict(state_dict)
    return model

## MAML adaptation

In [24]:
heldout_time = 0

In [25]:
model = loadModel(original_model_base+original_model, device="cuda", last=is_last)
#model = initModel(original_model_base+original_model, device="cuda")
original_params = {n: p for n, p in model.named_parameters()}

for day_id in support_batch.keys(): # tasks
    # if day_id not in heldout_dates:
    #     continue
    X_fs, y_fs, X_len_fs, y_len_fs = support_batch[day_id]
    fast_params = {n: p.clone() for n, p in original_params.items()}  
    
    def inner_loss(params, x, y, x_len, y_len):
            pred = functional_call(model, params, x)
            # CTC loss
            log_probs = torch.permute(pred.log_softmax(2), [1, 0, 2])  # (T, N, C)
            adjustedLens = ((x_len - model.kernelLen) / model.strideLen).to(torch.int32)
            ctc_loss = torch.nn.functional.ctc_loss(
                log_probs,  # (T, N, C)
                y,         # (N, S)
                adjustedLens,  # (N,)
                y_len,     # (N,)
                blank=0, reduction="mean", zero_infinity=True
            )
            return ctc_loss
    
    current = tm.time()
    # Inner Loop
    for _ in range(inner_step):
            loss = inner_loss(fast_params, X_fs, y_fs, X_len_fs, y_len_fs)
            grads = autograd_grad(loss,
                                  [p for p in fast_params.values() if p.requires_grad], 
                                  create_graph=not model.first_order)
            updated_params = {}
            trainable_keys = [k for k, p in fast_params.items() if p.requires_grad]
            for key, grad in zip(trainable_keys, grads):
                updated_params[key] = fast_params[key] - model.inner_lr * grad
            fast_params.update(updated_params)

    end = tm.time()
    if day_id in heldout_dates:
        time_per_sen = (end - current) / k_shot
        print(day_id, f"Time per sentence: {time_per_sen:.2f} seconds")
        heldout_time += time_per_sen

    torch.save(fast_params, save_path+day_id+"_weights.pth")

2023.04.17 Time per sentence: 0.92 seconds
2023.05.31 Time per sentence: 0.99 seconds
2023.06.28 Time per sentence: 1.30 seconds
2023.08.16 Time per sentence: 1.24 seconds
2023.10.09 Time per sentence: 1.23 seconds


In [26]:
avg_time = heldout_time / len(heldout_dates)
print(f"Inner steps: {inner_step}")
print(f"Average adaptation time per sentence: {avg_time:.2f} seconds")

Inner steps: 25
Average adaptation time per sentence: 1.13 seconds
